# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [ ]:
# 상위 폴더 path 설정
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from common_config import llm_connect

temperature = 0
max_tokens = 2086

llm = llm_connect()

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **LLM(대규모 언어 모델) 기반 애플리케이션을 “그래프(Graph)” 형태로 설계하고 실행하는 프레임워크**입니다.  
특히 **복잡한 에이전트 워크플로우**, **상태가 있는 대화**, **반복/분기 로직**, **여러 도구(tool) 호출** 같은 것을 체계적으로 만들 때 유용합니다.

## 한 줄로 요약
- **LangChain**이 LLM 앱을 “구성 요소 중심”으로 만들기 좋다면,
- **LangGraph**는 그 구성 요소들을 **노드(Node)와 엣지(Edge)**로 연결한 **상태 머신/그래프**처럼 제어하기 좋습니다.

---

## LangGraph에서 사용하는 핵심 개념

### 1) Graph(그래프)
애플리케이션의 흐름을 **그래프 구조**로 표현합니다.

- **노드(Node)**: 어떤 작업을 수행하는 단계  
  예: LLM 호출, 도구 실행, 검증, 요약
- **엣지(Edge)**: 다음에 어디로 갈지 결정하는 연결  
  예: “응답이 충분하면 종료”, “조건이 맞으면 검색 도구로 이동”

즉, “A 다음에는 B, 아니면 C” 같은 흐름을 명시적으로 표현합니다.

---

### 2) State(상태)
그래프를 돌면서 공유되는 **중간 데이터**입니다.

예:
- 대화 기록
- 사용자의 질문
- LLM의 응답
- 도구 실행 결과
- 플래닝 결과

LangGraph는 이 상태를 계속 업데이트하면서 다음 노드가 참고하게 합니다.

---

### 3) Node(노드)
실제 로직을 수행하는 함수나 컴포넌트입니다.

예:
- 사용자 입력 분석
- LLM에게 프롬프트 전달
- 검색 API 호출
- 결과 검증
- 최종 답변 생성

노드는 보통 **입력 상태를 받아서, 상태의 일부를 갱신한 결과를 반환**합니다.

---

### 4) Edge(엣지)
노드 간 이동 규칙입니다.

- **고정 엣지**: 항상 다음 노드로 이동
- **조건부 엣지**: 상태를 보고 다음 노드를 선택

예:
- 질문이

In [3]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 **LangChain의 확장**으로, **그래프 기반의 멀티 에이전트 워크플로우**를 구성할 수 있는 도구입니다.

사용하는 주요 개념은 다음과 같습니다:
- **State**: 작업 중 공유되는 상태
- **Node**: 개별 처리 단계
- **Edge**: 노드 간의 연결 및 흐름

이 개념들을 통해 복잡한 AI 파이프라인을 구현할 수 있습니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```